# EDA · Tanzania · perinatal (headline cohort)

Dataset: `data/processed/tz_perinatal.csv` — all pregnancies of 7+ months in the
last 5 years. Outcome: `perinatal_death` (stillbirth or death in the first 7 days).

Goal: understand the cohort and see which pregnancy-time factors separate the
pregnancies that end in a perinatal death from those that don't.


## 0 · Setup


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

DATA = '../../../data/processed/tz_perinatal.csv'
df = pd.read_csv(DATA)
df.shape


## 1 · First look


In [ ]:
df.head()


In [ ]:
df.info()


## 2 · Columns we care about

Separate the predictors (pregnancy-time info) from the survey-design columns and
the outcome, so the rest of the notebook stays focused.


In [ ]:
target = 'perinatal_death'
design = ['v001', 'v005', 'weight']   # cluster + survey weight, not predictors

predictors = [
    'maternal_age', 'prior_pregnancies', 'prior_live_births',
    'prior_stillbirth', 'prior_loss', 'preceding_interval_m',
    'multiple', 'education', 'wealth_quintile', 'residence',
    'region', 'told_high_bp',
]
numeric_preds = ['maternal_age', 'prior_pregnancies', 'prior_live_births',
                 'preceding_interval_m']
cat_preds = ['prior_stillbirth', 'prior_loss', 'multiple', 'education',
             'wealth_quintile', 'residence', 'told_high_bp']

df[predictors + [target]].describe(include='all').T


## 3 · Data quality


In [ ]:
missing = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
missing[missing > 0]


In [ ]:
print('duplicate rows:', df.duplicated().sum())
print('constant cols:', [c for c in df.columns if df[c].nunique(dropna=False) == 1])


`preceding_interval_m` is ~21% missing — these are **first pregnancies** (no
preceding birth), which is meaningful, not an error. We keep it and treat 'first
pregnancy' as its own signal rather than dropping the rows. `p6` (a raw DHS column)
is almost entirely empty and is not used.


## 4 · The outcome


In [ ]:
n = len(df)
events = int(df[target].sum())
print(f'perinatal deaths: {events} / {n} = {events/n:.2%}')
print('  stillbirths:', int(df['stillbirth'].sum()),
      '| early neonatal:', int(df['early_neonatal_death'].sum()))

w = df['v005'] / 1_000_000
print(f'weighted perinatal death rate: {(df[target]*w).sum()/w.sum():.2%}')


## 5 · Univariate — each variable on its own


In [ ]:
df[numeric_preds].hist(figsize=(11, 7), bins=30)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, c in zip(axes.ravel(), cat_preds):
    df[c].value_counts(dropna=False).plot(kind='bar', ax=ax, title=c)
for ax in axes.ravel()[len(cat_preds):]:
    ax.axis('off')
plt.tight_layout(); plt.show()


## 6 · Bivariate — perinatal death rate by each predictor

This is the core of the EDA: where does the death rate rise above the ~3.75% baseline?


In [ ]:
for c in cat_preds:
    print(f'\n--- {c} (death rate %) ---')
    print((df.groupby(c)[target].mean() * 100).round(2))


In [ ]:
# numeric predictors: bin into quintiles, then death rate per bin
for c in numeric_preds:
    binned = pd.qcut(df[c], 5, duplicates='drop')
    print(f'\n--- {c} (death rate %) ---')
    print((df.groupby(binned, observed=True)[target].mean() * 100).round(2))


In [ ]:
# clinically meaningful age / parity / interval cuts
age_grp = pd.cut(df['maternal_age'], [0, 18, 35, 100], right=False, labels=['<18', '18-34', '35+'])
par_grp = pd.cut(df['prior_live_births'], [-1, 0, 4, 100], labels=['0', '1-4', '5+'])
print('age group:', (df.groupby(age_grp, observed=True)[target].mean()*100).round(2).to_dict())
print('parity   :', (df.groupby(par_grp, observed=True)[target].mean()*100).round(2).to_dict())
di = df.dropna(subset=['preceding_interval_m'])
print('short interval (<24m):', round(di[di.preceding_interval_m < 24][target].mean()*100, 2),
      '| >=24m:', round(di[di.preceding_interval_m >= 24][target].mean()*100, 2))


In [ ]:
# visualise the strongest categorical splits
strong = ['multiple', 'prior_stillbirth', 'told_high_bp', 'prior_loss']
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
for ax, c in zip(axes, strong):
    (df.groupby(c)[target].mean()*100).plot(kind='bar', ax=ax, title=c)
    ax.axhline(df[target].mean()*100, color='red', ls='--', lw=1)
axes[0].set_ylabel('perinatal death rate (%)')
plt.tight_layout(); plt.show()


## 7 · Correlations (numeric)


In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df[numeric_preds + [target]].corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0)
plt.title('Numeric correlations'); plt.show()


## 8 · Findings

Baseline perinatal death rate in the Tanzania cohort is **3.75%** (412 of 10,980;
199 stillbirths + 213 early neonatal). The weighted rate is close, so the sample
is representative.

Factors that clearly raise risk above baseline:

- **Multiple pregnancy** — 13.7% vs 3.4%. The single strongest signal.
- **Previous stillbirth** — 9.8% vs 3.5%.
- **Told has high blood pressure** — 8.6% vs 3.5%. Supports the hypertension angle,
  though it's a self-report proxy.
- **Previous pregnancy loss** — 5.8% vs 3.4%.
- **Short birth interval (<24 months)** — 4.7% vs 3.1%.
- **Maternal age** — U-shaped: higher for <18 (4.3%) and 35+ (4.7%) than 18–34 (3.5%).
- **First pregnancy (parity 0)** — 4.9%, higher than parity 1–4 (3.2%).

Counter-intuitive and worth a caveat: risk is **higher** for richer (5.4% richest vs
2.5% poorest), urban (4.8% vs 3.4%) and more-educated mothers. This likely reflects
reporting/ascertainment — facility births are better recorded, so deaths are more
completely captured among wealthier, urban women — rather than wealth causing risk.
Flag this before using wealth or residence as predictors.

Data notes: `preceding_interval_m` missing = first pregnancies (keep as a flag);
`p6` unusable (97% empty). No duplicate rows.

For modelling: the history and clinical variables (multiple, prior stillbirth,
high BP, prior loss, short interval, age extremes) carry the signal and are all
genuinely pregnancy-time, so they are safe predictors.
